In [16]:
import os
import urllib.request
from ultralytics import YOLO
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

model = YOLO("yolo11n.pt")
print(f"\n모델 로드 완료: yolo11n.pt")

os.makedirs("outputs", exist_ok=True)
img_path = "outputs/bus.jpg"
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", img_path)
results = model(img_path)

for result in results:
    boxes = result.boxes
    print(f"검출된 객체 수: {len(boxes)}")

    for i, box in enumerate(boxes):
        cls_id = int(box.cls)
        cls_name = model.names[cls_id]
        conf = box.conf.item()
        xyxy = box.xyxy[0].tolist()

        print(f"객체 {i}: {cls_name} (conf={conf:.3f})")
        print(f"BBox: [{xyxy[0]:.1f}, {xyxy[1]:.1f},"
              f"{xyxy[2]:.1f}, {xyxy[3]:.1f}]")
        
    annotated = result.plot()
    cv2.imwrite("outputs/inference_result.jpg", annotated)
    print("\n결과 저장: outputs/inference_result.jpg")

for conf_thresh in [0.25, 0.50, 0.75]:
    results = model(img_path,
                    conf=conf_thresh, verbose=False)
    n_detections = len(results[0].boxes)
    print(f"conf={conf_thresh:.2f}: {n_detections}개 검출")


모델 로드 완료: yolo11n.pt

image 1/1 /workspace/study/physical-ai-study/Studies/Phase 3/week4/outputs/bus.jpg: 640x480 4 persons, 1 bus, 2.7ms
Speed: 1.2ms preprocess, 2.7ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 480)
검출된 객체 수: 5
객체 0: bus (conf=0.940)
BBox: [3.8, 229.4,796.2, 728.4]
객체 1: person (conf=0.888)
BBox: [671.0, 394.8,809.8, 878.7]
객체 2: person (conf=0.878)
BBox: [47.4, 399.6,239.3, 904.2]
객체 3: person (conf=0.856)
BBox: [223.1, 408.7,344.5, 860.4]
객체 4: person (conf=0.622)
BBox: [0.0, 556.1,68.9, 872.4]

결과 저장: outputs/inference_result.jpg
conf=0.25: 5개 검출
conf=0.50: 5개 검출
conf=0.75: 4개 검출


In [17]:
from ultralytics import YOLO
from ultralytics.utils import RUNS_DIR
import os

model = YOLO("yolo11n.pt")

# RUNS_DIR은 ultralytics 기본 저장 위치(physical-ai-study/runs)다. 그 부모인
# physical-ai-study를 기준으로 week4 경로를 조립하면 /workspace/study 같은 머신
# 종속 경로를 하드코딩하지 않아도 된다. 절대 경로라 runs/detect 중첩도 없다
project_dir = str(RUNS_DIR.parent / "Studies/Phase 3/week4/outputs/runs/detect")

results = model.train(
    data="coco128.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project=project_dir,
    name="coco128_baseline",
    exist_ok=True,  # 같은 이름 폴더에 덮어쓴다. 미지정 시 재실행마다 coco128_baseline-2처럼 자동 증가해, coco128_baseline을 하드코딩으로 읽는 실습 5 분석 셀이 옛 결과를 읽는다
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

# 경로를 재조립하지 말고 train이 실제로 저장한 디렉터리(results.save_dir)를 쓴다
result_dir = str(results.save_dir)

if os.path.exists(result_dir):
    files = os.listdir(result_dir)
    print(f"결과 디렉토리: {result_dir}")
    print(f"생성된 파일: {files}")

best_model = YOLO(f"{result_dir}/weights/best.pt")
metrics = best_model.val(project=project_dir, name="coco128_baseline_val", exist_ok=True)

print(f"\n mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")


print("\n결과 파일 확인:")
print(f"- {result_dir}/results.png (학습 커브)")
print(f"- {result_dir}/confusion_matrix.png (혼동 행렬)")
print(f"- {result_dir}/PR_curve.png (PR 커브)")

New https://pypi.org/project/ultralytics/8.4.58 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.56 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070, 11864MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coco128_

In [18]:
from ultralytics import YOLO
from ultralytics.utils import RUNS_DIR
import json
import os

project_dir = str(RUNS_DIR.parent / "Studies/Phase 3/week4/outputs/runs/detect")
os.makedirs("outputs", exist_ok=True)

experiments = [
    {
        "name": "exp_lr_low",
        "desc": "낮은 학습률 (lr0=0.001)",
        "params": {"lr0": 0.001, "epochs": 30}
    },
    {
        "name": "exp_lr_high",
        "desc": "높은 학습률 (lr0=0.02)",
        "params": {"lr0": 0.02, "epochs": 30}
    },
    {
        "name": "exp_no_mosaic",
        "desc": "Mosaic 비활성화",
        "params": {"mosaic": 0.0, "epochs": 30}
    },
    {
        "name": "exp_heavy_aug",
        "desc": "강한 Augmentation",
        "params": {"mosaic": 1.0, "mixup": 0.15, "degrees": 10.0,
                   "scale": 0.9, "epochs": 30}
    }
]

results_summary = []

for exp in experiments:
    print(f"실험: {exp['desc']}")

    model = YOLO("yolo11n.pt")

    try:
        result = model.train(
            data="coco128.yaml",
            imgsz=640,
            batch=16,
            device=0,
            project=project_dir,
            name=exp["name"],
            optimizer="SGD",  # optimizer=auto(기본값)는 lr0/momentum을 자체 계산해 덮어쓴다. lr 실험이 의미를 가지려면 옵티마이저를 명시해야 lr0가 실제로 적용된다
            exist_ok=True,  # 같은 이름 디렉터리에 덮어쓴다. 미지정 시 재실행마다 exp_lr_low-2처럼 자동 증가해 아래 평가가 옛 가중치를 읽는다
            patience=10,
            plots=True,
            verbose=False,
            **exp["params"]
        )

        # 경로 문자열을 재조립하지 말고 train이 실제로 저장한 디렉터리(result.save_dir)를 쓴다
        best_path = result.save_dir / "weights" / "best.pt"
        if best_path.exists():
            eval_model = YOLO(best_path)
            metrics = eval_model.val(verbose=False, project=project_dir, name=f"{exp['name']}_val", exist_ok=True)

            results_summary.append({
                "name": exp["name"],
                "desc": exp["desc"],
                "map50": metrics.box.map50,
                "map50_95": metrics.box.map,
                "precision": metrics.box.mp,
                "recall": metrics.box.mr,
            })
    except Exception as e:
        print(f"실험 실패: {e}")
        results_summary.append({
            "name": exp["name"],
            "desc": exp["desc"],
            "map50": 0,
            "map50_95": 0,
            "precision": 0,
            "recall": 0,
        })

print("실험 결과 비교")
print("=" * 70)
print(f"{'실험':20s} | {'mAP@0.5':>8s} | {'mAP@0.5:0.95':>12s} | {'Precision':>9s} | {'Recall':>6s}")
print("-" * 70)
for r in results_summary: 
    print(f"{r['desc']:20s} | {r['map50']:>8.4f} | {r['map50_95']:>12.4f} | "
          f"{r['precision']:>9.4f} | {r['recall']:>6.4f}")

with open('outputs/experiment_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)
print("\n결과 저장: outputs/experiment_results.json")

실험: 낮은 학습률 (lr0=0.001)
New https://pypi.org/project/ultralytics/8.4.58 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.56 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070, 11864MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_

In [19]:
import os
import yaml
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches

base_dir = "outputs/custom_dataset"
for split in ["train", "val"]:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)
print(f"{base_dir}/ 구조 생성 완료")

class_names = ["person", "car", "bicycle"]
np.random.seed(42)

def create_dummy_data(split, n_images):
    for i in range(n_images):
        img = np.random.randint(100, 200, (480, 640, 3), dtype=np.uint8)
        n_objects = np.random.randint(1, 5)
        labels = []

        for _ in range(n_objects):
            cls_id = np.random.randint(0, len(class_names))
            x1 = np.random.randint(10, 500)
            y1 = np.random.randint(10, 350)
            w = np.random.randint(40, 150)
            h = np.random.randint(40, 150)
            x2 = min(x1 + w, 639)
            y2 = min(y1 + h, 479)

            colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]
            cv2.rectangle(img, (x1, y1), (x2, y2), colors[cls_id], 2)

            x_center = (x1 + x2) / 2 / 640
            y_center = (y1 + y2) / 2 / 480
            width = (x2 - x1) / 640
            height = (y2 - y1) / 480

            labels.append(f"{cls_id} {x_center:.6f} {y_center:.6f} "
                          f"{width:.6f} {height:.6f}")
        img_name = f"img_{i:04d}.jpg"
        cv2.imwrite(f"{base_dir}/images/{split}/{img_name}", img)

        label_name = f"img_{i:04d}.txt"
        with open(f"{base_dir}/labels/{split}/{label_name}", "w") as f:
            f.write("\n".join(labels))

    print(f"{split}: {n_images}장 생성")

create_dummy_data("train", 20)
create_dummy_data("val", 5)

data_yaml = {
    "path": os.path.abspath(base_dir),
    "train": "images/train",
    "val": "images/val",
    "names": {i: name for i, name in enumerate(class_names)},
    "nc": len(class_names),
}

yaml_path = f"{base_dir}/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"저장: {yaml_path}")
print(f"내용:")
for key, value in data_yaml.items():
    print(f"{key}: {value}")

label_file = f"{base_dir}/labels/train/img_0000.txt"
print(f"라벨 파일: {label_file}")
with open(label_file, "r") as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        cls_id = int(parts[0])
        x_c, y_c, w, h = map(float, parts[1:])
        print(f"class={class_names[cls_id]}, "
              f"center=({x_c:.4f}, {y_c:.4f}), "
              f"size=({w:.4f}, {h:.4f})")
        
        assert 0 <= x_c <= 1, f"x_center 범위 오류: {x_c}"
        assert 0 <= y_c <= 1, f"y_center 범위 오류: {y_c}"
        assert 0 < w <= 1, f"width 범위 오류: {w}"
        assert 0 < h <= 1, f"height 범위 오류: {h}"

print("모든 좌표가 0~1 범위 내 (검증 통과)")

img = cv2.imread(f"{base_dir}/images/train/img_0000.jpg")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W = img.shape[:2]
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.imshow(img_rgb)

colors_plt = ["red", "lime", "blue"]
for line in lines:
    parts = line.strip().split()
    cls_id = int(parts[0])
    x_c, y_c, w, h = map(float, parts[1:])

    x1 = (x_c - w/2) * W
    y1 = (y_c - h/2) * H
    box_w = w * W
    box_h = h * H

    rect = patches.Rectangle(
        (x1, y1), box_w, box_h,
        linewidth=2, edgecolor=colors_plt[cls_id],
        facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, class_names[cls_id],
            fontsize=10, color=colors_plt[cls_id],
            fontweight="bold")
    
ax.set_title("YOLO Label Visualization")
plt.tight_layout()
plt.savefig("outputs/label_visualization.png", dpi=100)
print("저장: outputs/label_visualization.png")

outputs/custom_dataset/ 구조 생성 완료
train: 20장 생성
val: 5장 생성
저장: outputs/custom_dataset/data.yaml
내용:
path: /workspace/study/physical-ai-study/Studies/Phase 3/week4/outputs/custom_dataset
train: images/train
val: images/val
names: {0: 'person', 1: 'car', 2: 'bicycle'}
nc: 3
라벨 파일: outputs/custom_dataset/labels/train/img_0000.txt
class=car, center=(0.1836, 0.7604), size=(0.2328, 0.2375)
class=car, center=(0.5055, 0.4542), size=(0.1953, 0.2875)
class=car, center=(0.6086, 0.7573), size=(0.1922, 0.1562)
모든 좌표가 0~1 범위 내 (검증 통과)
저장: outputs/label_visualization.png


In [20]:
from ultralytics import YOLO
from ultralytics.utils import RUNS_DIR
import csv
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

project_dir = str(RUNS_DIR.parent / "Studies/Phase 3/week4/outputs/runs/detect")
result_dir = f"{project_dir}/coco128_baseline"
csv_path = f"{result_dir}/results.csv"
os.makedirs("outputs", exist_ok=True)

if os.path.exists(csv_path):
    epochs = []
    train_box_loss = []
    train_cls_loss = []
    val_map50 = []
    val_map50_95 = []

    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            epochs.append(int(row.get("epoch", row.get(" epoch", 0))))
            train_box_loss.append(float(row.get("train/box_loss", row.get(" train/box_loss", 0))))
            train_cls_loss.append(float(row.get("train/cls_loss", row.get(" train/cls_loss", 0))))
            val_map50.append(float(row.get("metrics/mAP50(B)", row.get(" metrics/mAP50(B)", 0))))
            val_map50_95.append(float(row.get("metrics/mAP50-95(B)", row.get(" metrics/mAP50-95(B)", 0))))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs, train_box_loss, "b-", label="Box Loss")
    axes[0].plot(epochs, train_cls_loss, 'r-', label='Cls Loss') # 분류 손실
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, val_map50, 'g-', label='mAP@0.5', linewidth=2)
    axes[1].plot(epochs, val_map50_95, 'b-', label='mAP@0.5:0.95', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('mAP')
    axes[1].set_title('Validation mAP')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].bar(['mAP@0.5', 'mAP@0.5:0.95'],
                [val_map50[-1], val_map50_95[-1]],
                color=['green', 'blue'], alpha=0.7)
    axes[2].set_ylabel('mAP')
    axes[2].set_title('Final Performance')
    axes[2].set_ylim(0, 1)
    for i, v in enumerate([val_map50[-1], val_map50_95[-1]]):
        axes[2].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=12)


    plt.tight_layout()
    plt.savefig('outputs/training_analysis.png', dpi=100)
    print("학습 커브 저장: outputs/training_analysis.png")
    print(f"최종 mAP@0.5: {val_map50[-1]:.4f}")
    print(f"최종 mAP@0.5:0.95: {val_map50_95[-1]:.4f}")
else:
    print(f"결과 파일 없음: {csv_path}")
    print("실습 2를 먼저 실행하세요.")

print(f"{'모델':10s} | {'파라미터':>10s} | {'mAP@0.5:0.95':>12s} | {'용도':12s}")
print("" + "-" * 55)
model_info = [
    ('YOLO11n', '2.6M', '39.5', 'Edge/실시간'),
    ('YOLO11s', '9.4M', '47.0', '경량 서버'),
    ('YOLO11m', '20.1M', '51.5', '균형'),
    ('YOLO11l', '25.3M', '53.4', '높은 정확도'),
    ('YOLO11x', '56.9M', '54.7', '최고 성능'),
]
for name, params, mAP, usage in model_info:
    print(f"{name:10s} | {params:>10s} | {mAP:>12s} | {usage:12s}")

학습 커브 저장: outputs/training_analysis.png
최종 mAP@0.5: 0.7944
최종 mAP@0.5:0.95: 0.6150
모델         |       파라미터 | mAP@0.5:0.95 | 용도          
-------------------------------------------------------
YOLO11n    |       2.6M |         39.5 | Edge/실시간    
YOLO11s    |       9.4M |         47.0 | 경량 서버       
YOLO11m    |      20.1M |         51.5 | 균형          
YOLO11l    |      25.3M |         53.4 | 높은 정확도      
YOLO11x    |      56.9M |         54.7 | 최고 성능       
